In [ ]:
# "id","title","vote_average","vote_count","status","release_date","revenue","runtime","budget","imdb_id","original_language","original_title","overview","popularity","tagline","genres","production_companies","production_countries","spoken_languages","cast","director","director_of_photography","writers","producers","music_composer","imdb_rating","imdb_votes","poster_path"

In [77]:
import pandas as pd
import os

In [78]:
DEST_FILE = "../data"
FILE_NAME = "TMDB_all_movies.csv"
full_path = os.path.join(DEST_FILE, FILE_NAME)

In [79]:
# Lecture du CSV
df = pd.read_csv(full_path)

In [80]:
# Tri
df_sorted = df.sort_values(
    by=["vote_count", "popularity", "vote_average"],
    ascending=[False, False, False]
)

In [81]:
# Garder seulement les 50000 premiers
df_top = df_sorted.head(50000)

In [82]:
top_path = os.path.join(DEST_FILE, "TMDB_top.csv")

In [83]:
if os.path.exists(top_path):
    os.remove(top_path)

In [84]:
df_top.to_csv(top_path, index=False)

In [85]:
df = df_top.copy()

In [86]:
# Conversions numériques
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").astype(float)
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").astype("Int64")  # Int64 pour accepter NaN
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce").astype(float)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce").astype(float)
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce").astype(float)

In [87]:
# Dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

In [88]:
# Colonnes à transformer en listes
array_columns = [
    "genres", "production_countries", "production_companies", "cast", "director", "writers"
]

for col in array_columns:
    df[col + "_array"] = (
        df[col].str.split(r",\s*")       # découper sur virgule + espace
             .apply(lambda x: x if isinstance(x, list) else [])  # remplacer NaN par []
    )

# Suppression des colonnes originales
df = df.drop(columns=array_columns)

In [89]:
# Vérifier les types
print(df.dtypes)

id                                     int64
title                                 object
vote_average                         float64
vote_count                             Int64
status                                object
release_date                  datetime64[ns]
revenue                              float64
runtime                              float64
budget                               float64
imdb_id                               object
original_language                     object
original_title                        object
overview                              object
popularity                           float64
tagline                               object
spoken_languages                      object
director_of_photography               object
producers                             object
music_composer                        object
imdb_rating                          float64
imdb_votes                           float64
poster_path                           object
release_ye

In [90]:
def count_empty_values(df):
    counts = {}
    for col in df.columns:
        counts[col] = (
            df[col].isna()                                  # NaN / None
            | (df[col] == "")                               # chaîne vide
            | (df[col].apply(lambda x: isinstance(x, list) and len(x) == 0))  # liste vide
        ).sum()
    return pd.Series(counts, name="empty_count")

In [102]:
empty_counts = count_empty_values(df)
print(empty_counts)

id                               0
title                            0
vote_average                     0
vote_count                       0
release_date                    10
runtime                          0
budget                           0
original_title                   0
overview                         0
popularity                       0
poster_path                    136
release_year                     0
genres_array                    77
production_countries_array     755
production_companies_array    2370
cast_array                     620
director_array                 151
writers_array                 2423
Name: empty_count, dtype: int64


In [92]:
df = df.drop(
    ["status", "imdb_id", "tagline", "director_of_photography",
     "producers", "imdb_rating", "imdb_votes",
     "music_composer", "revenue", "spoken_languages", "original_language"],
    axis=1
)

In [93]:
df.head(100)

,id,title,vote_average,vote_count,release_date,runtime,budget,original_title,overview,popularity,poster_path,release_year,genres_array,production_countries_array,production_companies_array,cast_array,director_array,writers_array
16326,27205,Inception,8.369,37762,2010-07-15,148.0,160000000.0,Inception,"Cobb, a skilled thief who commits corporate es...",28.2298,/ljsZTbVsrQSqZgWeep2B1QiDKuh.jpg,2010,"[Action, Science Fiction, Adventure]",[United States of America],"[Legendary Pictures, Syncopy, Warner Bros. Pic...","[Jason Tendell, Johnathan Geare, Natasha Beaum...",[Christopher Nolan],[Christopher Nolan]
96467,157336,Interstellar,8.458,37611,2014-11-05,169.0,165000000.0,Interstellar,The adventures of a group of explorers who mak...,40.6050,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,2014,"[Adventure, Drama, Science Fiction]","[United Kingdom, United States of America]","[Legendary Pictures, Syncopy, Lynda Obst Produ...","[Francis X. McCarthy, Leah Cairns, William Pat...",[Christopher Nolan],"[Christopher Nolan, Jonathan Nolan]"
116,155,The Dark Knight,8.523,34173,2008-07-16,152.0,185000000.0,The Dark Knight,Batman raises the stakes in his war on crime. ...,24.2977,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,2008,"[Drama, Action, Crime, Thriller]","[United Kingdom, United States of America]","[Warner Bros. Pictures, Legendary Pictures, Sy...","[Nestor Carbonell, Chris Wilson, Rob Clark, Da...",[Christopher Nolan],"[Jonathan Nolan, Bob Kane, Christopher Nolan, ..."
14426,24428,The Avengers,7.785,32623,2012-04-25,143.0,220000000.0,The Avengers,When an unexpected enemy emerges and threatens...,33.6308,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,2012,"[Science Fiction, Action, Adventure]",[United States of America],[Marvel Studios],"[Holly Neelie, Brent McGee, Stan Lee, Josh Cow...",[Joss Whedon],"[Zak Penn, Stan Lee, Jack Kirby, Joss Whedon, ..."
12044,19995,Avatar,7.592,32470,2009-12-15,162.0,237000000.0,Avatar,"In the 22nd century, a paraplegic Marine is di...",25.6993,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,2009,"[Action, Adventure, Fantasy, Science Fiction]","[United States of America, United Kingdom]","[Dune Entertainment, Lightstorm Entertainment,...","[Debra Wilson, Julene Renee, Alicia Vela-Baile...",[James Cameron],[James Cameron]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116263,198663,The Maze Runner,7.192,17471,2014-09-10,113.0,34000000.0,The Maze Runner,"Set in a post-apocalyptic world, young Thomas ...",23.8984,/ode14q7WtDugFDp78fo9lCsmay9.jpg,2014,"[Action, Mystery, Science Fiction, Thriller]","[United States of America, United Kingdom]","[Ingenious Media, The Gotham Group, Dayday Fil...","[Andrew Varenhorst, SanChavis Torns, Lane West...",[Wes Ball],"[James Dashner, Grant Pierce Myers, Noah Oppen..."
119772,205596,The Imitation Game,7.993,17452,2014-11-14,113.0,14000000.0,The Imitation Game,Based on the real life story of legendary cryp...,11.1699,/zSqJ1qFq8NXFfi7JeIYMlzyR0dx.jpg,2014,"[History, Drama, Thriller, War]","[United States of America, United Kingdom]","[Bristol Automotive, Black Bear Pictures, Film...","[Josh Wichard, Tim Steed, Andrew Havill, Charl...",[Morten Tyldum],"[Andrew Hodges, Graham Moore]"
195020,313369,La La Land,7.899,17417,2016-12-01,129.0,30000000.0,La La Land,"Mia, an aspiring actress, serves lattes to mov...",9.6792,/uDO8zWDhfWwoFdKS4fzkUJt0Rf0.jpg,2016,"[Comedy, Drama, Romance, Music]",[United States of America],"[Summit Entertainment, Gilbert Films, Impostor...","[Carol Connors, Megan Lawson, Gustavo Vargas, ...",[Damien Chazelle],"[Justin Paul, Benj Pasek, Damien Chazelle]"
11336,18785,The Hangover,7.325,17387,2009-06-02,100.0,35000000.0,The Hangover,When three friends finally come to after a rau...,17.4951,/A0uS9rHR56FeBtpjVki16M5xxSW.jpg,2009,[Comedy],[United States of America],"[Legendary Pictures, Green Hat Films, Warner B...","[Matt Walsh, Michael Li, Carrot Top, Chuck Pac...",[Todd Phillips],"[Jon Lucas, Scott Moore]"


In [ ]:
# Delete line with empty title
df = df[df["title"].notna() & (df["title"] != "")]

In [99]:
# Delete line with empty overview
df = df[df["overview"].notna() & (df["overview"] != "")]

In [101]:
df = df.fillna({
    'release_year': -1
})

In [103]:
df.shape

(49782, 18)